## Part 2: Query Decomposition and Dynamic Filtering

A query like "show me black formal dresses that aren't in the basics section" has multiple semantic components. No single retrieval call will satisfy it reliably.

In this notebook, we build agents that generate dynamic Qdrant filters from natural language and decompose complex multi-part queries into discrete subqueries.

We'll use the `products` collection (2,500 products) from Part 1.

**Key concepts**: query decomposition, subquery generation, dynamic filtering, parallel retrieval, result unification.

## Setup

In [ ]:
#!pip install pydantic-ai

In [ ]:
import os
import openai
from dotenv import load_dotenv

load_dotenv()

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    print("No OpenAI API key found. Update your .env file.")
else:
    openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)

    def embed_text(text: str) -> list[float]:
        response = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding

    print(f"Qdrant URL: {QDRANT_URL}")
    print(f"OpenAI API Key: {OPENAI_API_KEY[:10]}...")

## Reconnect to Qdrant

In [ ]:
from qdrant_client import QdrantClient

qdrant = QdrantClient(url=QDRANT_URL)

COLLECTION_NAME = "products"
count = qdrant.count(collection_name=COLLECTION_NAME).count
print(f"Connected. Collection '{COLLECTION_NAME}' has {count} points")

In [ ]:
# Check what payload fields are available
sample = qdrant.scroll(collection_name=COLLECTION_NAME, limit=1)[0]
payload_fields = list(sample[0].payload.keys())
print(f"Payload fields: {payload_fields}")

## Build a Dynamic Filtering Agent with Tools

Instead of hard-coding the valid filter values, we will give the agent a **tool** that lets it inspect the collection at runtime. The agent decides which fields to look up, calls the tool to get the actual values from the database, and then builds precise filters.

This is more robust (works with any dataset) and demonstrates a core concept in agentic AI: **agents that gather information before acting**.

In [ ]:
from pydantic_ai import Agent, Tool
from qdrant_client.models import Filter, FieldCondition, MatchValue, MatchAny

# This is the tool the agent will use to inspect the collection
def get_unique_values(field_name: str) -> str:
    """Look up the unique values for a payload field in the products collection.
    Use this to find the exact values available for filtering before creating filters.
    For example, call this with 'colour_group_name' to see all available colors,
    or 'index_name' to see all available categories."""
    all_points, _ = qdrant.scroll(collection_name=COLLECTION_NAME, limit=2500, with_payload=True)
    values = sorted(set(p.payload.get(field_name, "") for p in all_points if p.payload.get(field_name)))
    return f"Field '{field_name}' has these values: {values}"

# Let's test it ourselves first
print(get_unique_values("index_name"))
print()
print(get_unique_values("colour_group_name"))

In [ ]:
smart_filter_agent = Agent(
    model="openai:gpt-4.1-mini",
    instructions=f"""
    You analyze user queries and create Qdrant filters for an e-commerce product database.
    Available payload fields: {payload_fields}
    
    BEFORE generating any filter, you MUST use the get_unique_values tool to look up
    the actual values in the database for any field you plan to filter on.
    Do NOT guess values — always check first.
    
    For example, if the user says "women's tops", call get_unique_values("index_name") 
    to find that women's products are under "Ladieswear", not "Women" or "Womens".
    
    After looking up values, generate Python code that creates a Qdrant Filter object.
    Only return the filter code as your final answer, nothing else.
    
    Use these Qdrant filter classes (already imported):
    - Filter(must=[...]) for AND conditions
    - Filter(should=[...]) for OR conditions  
    - Filter(must_not=[...]) for NOT conditions
    - FieldCondition(key="field_name", match=MatchValue(value="exact_value"))
    - FieldCondition(key="field_name", match=MatchAny(any=["val1", "val2"])) for matching multiple values
    
    If no filters needed, return: None
    """,
    tools=[Tool(get_unique_values)],
)

async def auto_filtered_search(user_query):
    filter_result = await smart_filter_agent.run(f"Create filters for: {user_query}")
    filter_code = filter_result.output.strip()

    print(f"Query: {user_query}")
    print(f"Generated filter: {filter_code}")

    query_filter = None
    if filter_code != "None":
        try:
            query_filter = eval(filter_code)
        except Exception as e:
            print(f"Filter creation failed: {e}, searching without filters")

    query_vector = embed_text(user_query)
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=query_filter,
        limit=5,
    ).points

    print(f"\nFound {len(results)} products:")
    for i, hit in enumerate(results, 1):
        print(f"  {i}. {hit.payload['prod_name']} ({hit.payload['colour_group_name']}, {hit.payload['section_name']})")

    return results

In [ ]:
results = await auto_filtered_search("Men's dark colored jackets, not black")

## Combining Query Optimization with Dynamic Filtering

Now let's build an agent that combines both techniques: optimizing the search query AND dynamically generating filters.

In [ ]:
combined_agent = Agent(
    model="openai:gpt-4.1-mini",
    instructions=f"""
    You are an intelligent product search assistant that performs TWO tasks:
    
    1. OPTIMIZE the user's natural language query for better vector search results
    2. GENERATE appropriate Qdrant filters based on the query context
    
    Available payload fields: {payload_fields}
    
    BEFORE generating any filter, you MUST use the get_unique_values tool to look up
    the actual values in the database for any field you plan to filter on.
    Do NOT guess values — always check first.
    
    Return your response in this exact format:
    OPTIMIZED_QUERY: <the optimized search query>
    FILTERS: <Python Qdrant filter code or None>
    
    Filter syntax:
    - Filter(must=[FieldCondition(key="field", match=MatchValue(value="exact_value"))])
    - Filter(must_not=[FieldCondition(key="field", match=MatchValue(value="exact_value"))])
    - FieldCondition(key="field", match=MatchAny(any=["val1", "val2"])) for multiple values
    """,
    tools=[Tool(get_unique_values)],
)

async def smart_search(user_query):
    print(f"Original query: '{user_query}'")
    print("=" * 70)
    
    agent_result = await combined_agent.run(f"Process this query: {user_query}")
    agent_output = agent_result.output
    
    lines = agent_output.strip().split('\n')
    optimized_query = None
    filter_code = None
    
    for line in lines:
        if line.startswith('OPTIMIZED_QUERY:'):
            optimized_query = line.replace('OPTIMIZED_QUERY:', '').strip()
        elif line.startswith('FILTERS:'):
            filter_code = line.replace('FILTERS:', '').strip()
    
    print(f"Optimized query: '{optimized_query}'")
    print(f"Generated filters: {filter_code}")
    print("=" * 70)
    
    query_filter = None
    if filter_code and filter_code != "None":
        try:
            query_filter = eval(filter_code)
        except Exception as e:
            print(f"Filter generation failed: {e}")
    
    query_vector = embed_text(optimized_query if optimized_query else user_query)
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=query_filter,
        limit=5,
    ).points
    
    print(f"\nFound {len(results)} products:\n")
    for i, hit in enumerate(results, 1):
        print(f"{i}. {hit.payload['prod_name']}")
        print(f"   Type: {hit.payload['product_type_name']} | Color: {hit.payload['colour_group_name']}")
        print(f"   Section: {hit.payload['section_name']} | Score: {hit.score:.4f}")
        print(f"   {hit.payload['detail_desc'][:120]}...")
        print()
    
    return results

### Example 1: Complex natural language query

In [ ]:
results = await smart_search(
    "I'm looking for some comfy casual stuff for women, maybe in lighter colors, nothing too formal"
)

### Example 2: Query with implicit filtering needs

In [ ]:
results = await smart_search(
    "Show me children's sportswear, no pink"
)

## Query Decomposition Agent

Now let's tackle multi-part queries. A question like "compare the men's jacket options with the women's coat options in dark colors" requires two separate retrieval calls with different filters.

In [ ]:
import json

decomposition_agent = Agent(
    model="openai:gpt-4.1-mini",
    instructions=f"""
    You decompose complex user queries into discrete subqueries for a product vector database.
    
    Available payload fields: {payload_fields}
    
    BEFORE generating any filter, you MUST use the get_unique_values tool to look up
    the actual values in the database for any field you plan to filter on.
    Do NOT guess values — always check first.
    
    For each subquery, return a JSON array where each element has:
    - "query": the optimized search query for this subquery
    - "filters": Python Qdrant filter code string (or null)
    - "purpose": what this subquery is looking for
    
    Return ONLY valid JSON as your final answer. No explanation.
    
    Qdrant filter syntax:
    - Filter(must=[FieldCondition(key="field", match=MatchValue(value="val"))])
    - FieldCondition(key="field", match=MatchAny(any=["val1", "val2"])) for multiple values
    """,
    tools=[Tool(get_unique_values)],
)

async def decomposed_search(user_query):
    print(f"Original query: '{user_query}'")
    print("=" * 70)
    
    result = await decomposition_agent.run(f"Decompose this query: {user_query}")
    subqueries = json.loads(result.output)
    
    print(f"Decomposed into {len(subqueries)} subqueries:\n")
    
    all_results = []
    for i, sq in enumerate(subqueries, 1):
        print(f"Subquery {i}: {sq['purpose']}")
        print(f"  Search: '{sq['query']}'")
        print(f"  Filters: {sq['filters']}")
        
        query_filter = None
        if sq['filters'] and sq['filters'] != 'null':
            try:
                query_filter = eval(sq['filters'])
            except Exception as e:
                print(f"  Filter failed: {e}")
        
        query_vector = embed_text(sq['query'])
        results = qdrant.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            query_filter=query_filter,
            limit=3,
        ).points
        
        print(f"  Found {len(results)} results")
        for hit in results:
            print(f"    - {hit.payload['prod_name']} ({hit.payload['colour_group_name']}, {hit.payload['section_name']})")
        print()
        
        all_results.extend(results)
    
    return all_results

In [ ]:
results = await decomposed_search(
    "Compare the men's dark colored jacket options with women's red or pink dresses"
)

## Key Takeaways

**What we've built:**
1. **Dynamic filtering agent** — Automatically generates Qdrant payload filters from natural language
2. **Combined optimization + filtering** — Query optimization AND filter generation in one agent
3. **Query decomposition agent** — Breaks multi-part queries into targeted subqueries with individual filters

**The agent advantage:**
- **Before agents**: You'd need to manually write filter code, optimize queries, and chain operations
- **With agents**: Natural language in, optimized database queries with filters out — automatically

Next up: we take this from single-turn RAG to **conversational AI** with intent classification and multi-agent orchestration.

In [ ]:
qdrant.close()